In [ ]:
import torch
import torch.nn as nn

dtype = torch.float16

# Hyperparameters
d_batch = 4
d_context = 128
d_embed_features = 256
d_head_count = 2
d_head_features = d_embed_features // d_head_count  # 128
d_vocab = 4069

# Tensor shapes
s_tokens = (d_batch, d_context)  # (4, 128)
s_vocab = (d_vocab, d_embed_features)  # (4069, 256)
s_batch = (d_batch, d_context, d_embed_features)  # (4, 128, 256)
s_weight = (d_embed_features, d_embed_features)  # (256, 256)
s_qkv = (d_batch, d_context, d_head_count, d_head_features)  # (4, 128, 2, 128)

# 0. Vocabulary
vocab = nn.Parameter(torch.randn(s_vocab, dtype=dtype))  # (4069, 256)

# 1. Input
tokens = torch.randint(0, d_vocab, s_tokens, dtype=torch.long)  # (4, 128)
batch = vocab[tokens]  # (4, 128, 256)

# 2. Projection Weights
wq = nn.Parameter(torch.randn(s_weight, dtype=dtype))  # (256, 256)
wk = nn.Parameter(torch.randn(s_weight, dtype=dtype))  # (256, 256)
wv = nn.Parameter(torch.randn(s_weight, dtype=dtype))  # (256, 256)

# 3. Q, K, V Projections
q = torch.matmul(batch, wq).reshape(s_qkv).transpose(1, 2)  # (4, 2, 128, 128)
k = torch.matmul(batch, wk).reshape(s_qkv).transpose(1, 2)  # (4, 2, 128, 128)
v = torch.matmul(batch, wv).reshape(s_qkv).transpose(1, 2)  # (4, 2, 128, 128)

# 4. Attention Scores
scores = torch.matmul(q, k.transpose(-2, -1)) / (d_head_features ** 0.5)  # (4, 2, 128, 128)
attn = torch.softmax(scores, dim=-1)  # (4, 2, 128, 128)

# 5. Attention Output
out = torch.matmul(attn, v)  # (4, 2, 128, 128)
out = out.transpose(1, 2).reshape(s_batch)  # (4, 128, 256)

# 6. Final Output Projection (W_O)
wo = nn.Parameter(torch.randn(s_weight, dtype=dtype))  # (256, 256)
out = torch.matmul(out, wo)  # (4, 128, 256)

print(out.shape)  # torch.Size([4, 128, 256])

torch.Size([4, 128, 256])
